# Jackfruit H3: colliders with activation patching
Rule in every story: **C happens if A or B** (for example, the grass gets wet if it rains or if the sprinkler is on). We patch the state of the C word ("the grass was **green**" receives the state of "**wet**") and ask about **B** (did it rain?).

* **Seeing C** raises belief in B. **Doing C** (a neighbour poured water on the grass) should not.
* **Explaining away:** once C is known, learning A (the sprinkler was on) should *lower* belief in B.
  Interaction = [B(C) − B(C and A)] − [B(no C) − B(A only)]. It is positive if A lowers B more when C has been established.

**Questions:** (1) Does a C patch act like seeing or doing (D, as for chains, with C's own question as the manipulation check)? (2) Does a C patch produce explaining away, as reading C does?

**Pass rule (fixed before running, same as chains):** the true patch moves the filler-corrected C answer by ≥ 1.0, beats the unrelated patch by > 0.5 in ≥ 70% of scenarios, the filler moves < 0.5, and the false patch moves C below 0.
**Story filters:** ΔB(see) > 0.5, ΔC(see) > 0.5, ΔC(do) > 0.5, r_see − r_do > 0.25, with r = ΔB / ΔC.
Word patches only (the chain runs showed word and sentence patches agree). T4, about 25 minutes. **Runtime → Run all.**

In [ ]:
# Mount Google Drive and work inside the jackfruit folder (story files live there; results are saved there)
import os
from google.colab import drive
drive.mount("/content/drive")
os.chdir("/content/drive/MyDrive/jackfruit")
OUT = "results"
os.makedirs(OUT, exist_ok=True)
print("working in", os.getcwd(), "| files:", sorted(os.listdir(".")))

In [ ]:
# ---------- Setup: model, tokenizer, settings
import gc, os, time, torch, numpy as np, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
torch.set_grad_enabled(False)

NAME = "Qwen/Qwen3-4B-Base"
DEVICE = "cuda"
BS = 16                         # batch size; lower to 8 if you get out-of-memory errors
TRAIN_VARIANTS = range(0, 16)   # used only to learn directions
EVAL_VARIANTS = range(16, 20)   # held out; used for every score. Raise to range(16, 24) for the final run
LAYER_STEP = 3
ALPHAS = [1.0, 2.0, 4.0]        # edit strength, in units of the (true - false) difference vector
POSITIONS = ["mediator", "after", "answer"]

for v in ["lm", "model", "W32"]:
    if v in globals():
        del globals()[v]
gc.collect(); torch.cuda.empty_cache()

tok = AutoTokenizer.from_pretrained(NAME)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def load(dtype):
    m = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype=dtype, device_map=DEVICE).eval()
    return m, m.get_output_embeddings().weight.float()     # float32 output layer

lm, W32 = load(torch.float16)
N_LAYERS = lm.config.num_hidden_layers
LAYERS = list(range(0, N_LAYERS, LAYER_STEP))
print("layers to sweep:", LAYERS)

YES = tok(" Yes", add_special_tokens=False).input_ids
NO = tok(" No", add_special_tokens=False).input_ids
assert len(YES) == 1 and len(NO) == 1
YES, NO = YES[0], NO[0]

In [ ]:
# ---------- Prompt building, token positions, belief with optional edit
import ast
FEWSHOT = (
    "Story: The kettle was on the stove and the water started to bubble.\n"
    "Question: Was the water hot?\nAnswer: Yes\n\n"
    "Story: Maya left her umbrella at home and the sky stayed clear all day.\n"
    "Question: Did Maya get wet from rain?\nAnswer: No\n\n"
)
STORY_OFFSET = len(FEWSHOT) + len("Story: ")

def prompt(story, q):
    return f"{FEWSHOT}Story: {story}\nQuestion: {q}\nAnswer:"

def encode(stories, qs, spans):
    """Tokenise a batch (left-padded) and find, per row, the last token of the key sentence."""
    enc = tok([prompt(s, q) for s, q in zip(stories, qs)], return_tensors="pt", padding=True,
              return_offsets_mapping=True)
    offs = enc.pop("offset_mapping")
    am = enc["attention_mask"]
    key_idx = []
    for i, (a, b) in enumerate(spans):
        a, b = a + STORY_OFFSET, b + STORY_OFFSET
        hit = ((offs[i, :, 0] < b) & (offs[i, :, 1] > a) & (am[i] == 1)).nonzero().flatten()
        assert len(hit) > 0, "key sentence not found in tokens"
        key_idx.append(hit[-1].item())
    return enc, torch.tensor(key_idx)

def position_mask(key_idx, T, mode):
    B = len(key_idx)
    m = torch.zeros(B, T)
    for i, k in enumerate(key_idx.tolist()):
        if mode == "mediator":
            m[i, k] = 1
        elif mode == "after":
            m[i, k + 1:T - 1] = 1     # tokens after the key sentence, excluding the final token
        elif mode == "answer":
            m[i, T - 1] = 1
    return m

def _layer_out(out):
    return out[0] if isinstance(out, tuple) else out

def belief(stories, qs, spans, edit=None, bs=BS):
    """log P(Yes) - log P(No). edit = dict(layer=int, vecs=[N, H] tensor, mode=str) or None."""
    res = []
    for i in range(0, len(stories), bs):
        enc, key_idx = encode(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        handle = None
        if edit is not None:
            T = enc["input_ids"].shape[1]
            mask = position_mask(key_idx, T, edit["mode"]).to(DEVICE, lm.dtype)
            vecs = edit["vecs"][i:i+bs].to(DEVICE, lm.dtype)
            def hook(mod, inp, out, mask=mask, vecs=vecs):
                h = _layer_out(out) + mask[:, :, None] * vecs[:, None, :]
                return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
            handle = lm.model.layers[edit["layer"]].register_forward_hook(hook)
        try:
            h = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                         position_ids=pos).last_hidden_state[:, -1].float()
        finally:
            if handle is not None:
                handle.remove()
        logp = torch.log_softmax(h @ W32.T, dim=-1)
        res += (logp[:, YES] - logp[:, NO]).tolist()
    return np.array(res)

def key_states(stories, qs, spans, layers, bs=BS):
    """Residual stream (output of each layer in `layers`) at the last token of the key sentence."""
    out = []
    for i in range(0, len(stories), bs):
        enc, key_idx = encode(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        hs = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos,
                      output_hidden_states=True).hidden_states
        rows = torch.arange(len(key_idx))
        out.append(torch.stack([hs[L + 1][rows, key_idx.to(DEVICE)].float().cpu() for L in layers], 1))
    return torch.cat(out)        # [N, len(layers), H]

# float16 overflow check; fall back to bfloat16 if needed
t = belief(["Priya ate a huge lunch and felt full."], ["Was Priya full?"], [[0, 5]])
if not np.isfinite(t).all():
    print("float16 overflow -> bfloat16")
    del lm, W32; gc.collect(); torch.cuda.empty_cache()
    lm, W32 = load(torch.bfloat16)
print("dtype:", lm.dtype)

d = pd.read_csv("stories_chain.csv")
d["variant"] = d.item_id.str.rsplit("_", n=1).str[1].astype(int)
for c in [c for c in d.columns if c.startswith("span_")]:
    d[c] = d[c].apply(ast.literal_eval)
train = d[d.variant.isin(TRAIN_VARIANTS)].reset_index(drop=True)
ev = d[d.variant.isin(EVAL_VARIANTS)].reset_index(drop=True)
SCEN = sorted(d.scenario_id.unique())
print(f"{len(SCEN)} scenarios | train items {len(train)} | eval items {len(ev)}")

In [ ]:
import glob, matplotlib.pyplot as plt
from scipy.stats import wilcoxon
TAG, EVAL, MODE = "H3", range(16, 24), "word"
KINDS = ["true", "false", "unrelated"]
FILLERS = ["Was there a cat nearby?", "Did someone in the story have a brother?", "Was a radio playing?"]
P = pd.read_csv("pairs_collider.csv")
for c in [c for c in P.columns if c.startswith(("span_", "wspan_"))]:
    P[c] = P[c].apply(ast.literal_eval)
P = P[P.variant.isin(EVAL)].reset_index(drop=True)
for i, f in enumerate(FILLERS):
    P[f"q_f{i}"] = f
PROBES = {"b": "q_b", "c": "q_c", **{f"f{i}": f"q_f{i}" for i in range(len(FILLERS))}}
print(f"{P.scenario_id.nunique()} scenarios, {len(P)} items | layers {LAYERS}")

## Step 1: the model's own seeing, doing and explaining away in text

In [ ]:
path = f"{OUT}/{TAG}_reference_items.csv"
if os.path.exists(path):
    ref = pd.read_csv(path)
else:
    ref = P[["scenario_id", "item_id"]].copy()
    for cond in ["base", "see", "do", "false", "see_A", "A_only"]:
        for p, q in PROBES.items():
            ref[f"{p}_{cond}"] = belief(P[cond].tolist(), P[q].tolist(), P[f"span_{cond}"].tolist())
    ref.to_csv(path, index=False)
nf = len(FILLERS)
fillm = lambda cond: sum(ref[f"f{i}_{cond}"] for i in range(nf)) / nf
for cond in ["see", "do", "false", "see_A", "A_only"]:
    for p in ["b", "c"]:
        ref[f"d{p}C_{cond}"] = (ref[f"{p}_{cond}"] - fillm(cond)) - (ref[f"{p}_base"] - fillm("base"))
ref["EA_text"] = (ref.dbC_see - ref.dbC_see_A) - (0 - ref.dbC_A_only)
sc = ref.groupby("scenario_id")[[c for c in ref.columns if c.startswith(("d", "EA"))]].mean()
sc["r_see"] = sc.dbC_see / sc.dcC_see
sc["r_do"] = sc.dbC_do / sc.dcC_do
sc["valid"] = (sc.dbC_see > 0.5) & (sc.dcC_see > 0.5) & (sc.dcC_do > 0.5) & (sc.r_see - sc.r_do > 0.25)
sc.to_csv(f"{OUT}/{TAG}_reference_scenarios.csv")
display(sc[["dcC_see", "dbC_see", "dcC_do", "dbC_do", "dbC_A_only", "EA_text", "r_see", "r_do", "valid"]].round(2))
VALID = sc.index[sc.valid].tolist()
print(f"valid: {len(VALID)} / {len(sc)} | text explaining-away interaction, median over valid: {sc.loc[VALID, 'EA_text'].median():.2f}")

## Step 2: patch the C word (base story, and the story where A is stated)

In [ ]:
def key_positions(stories, qs, spans, wspans):
    """Per row: index of the last token of the key word, and of the sentence end."""
    enc = tok([prompt(s, q) for s, q in zip(stories, qs)], return_tensors="pt", padding=True, return_offsets_mapping=True)
    offs = enc.pop("offset_mapping"); am = enc["attention_mask"]
    wl, se = [], []
    for i, ((a, b), (wa, wb)) in enumerate(zip(spans, wspans)):
        def last_hit(x, y):
            x, y = x + STORY_OFFSET, y + STORY_OFFSET
            h = ((offs[i, :, 0] < y) & (offs[i, :, 1] > x) & (am[i] == 1)).nonzero().flatten()
            assert len(h) > 0
            return h[-1].item()
        wl.append(last_hit(wa, wb)); se.append(last_hit(a, b))
    return enc, wl, se

def record(stories, qs, spans, wspans, bs=BS):
    """States at [key-word-last ... sentence-end] for every layer in LAYERS. Returns list of [L, n_pos, H] tensors."""
    out = []
    for i in range(0, len(stories), bs):
        enc, wl, se = key_positions(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs], wspans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        hs = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos,
                      output_hidden_states=True).hidden_states
        for r in range(len(wl)):
            out.append(torch.stack([hs[L + 1][r, wl[r]:se[r] + 1].float().cpu() for L in LAYERS]))
    return out

def belief_patch(stories, qs, spans, wspans, src, li, mode, bs=BS):
    """Run the target stories with states replaced at layer LAYERS[li]. src[row] = [L, n_pos, H]."""
    res, skipped = [], 0
    L = LAYERS[li]
    for i in range(0, len(stories), bs):
        enc, wl, se = key_positions(stories[i:i+bs], qs[i:i+bs], spans[i:i+bs], wspans[i:i+bs])
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
        items = []
        for r in range(len(wl)):
            s_states = src[i + r][li]                         # [n_pos_src, H]
            n_tgt = se[r] - wl[r] + 1
            if mode == "word":
                items.append(([wl[r]], s_states[:1]))
            elif s_states.shape[0] == n_tgt:
                items.append((list(range(wl[r], se[r] + 1)), s_states))
            else:
                items.append(None); skipped += 1
        def hook(mod, inp, out, items=items):
            h = _layer_out(out).clone()
            for r, it in enumerate(items):
                if it is not None:
                    h[r, it[0]] = it[1].to(h.device, h.dtype)
            return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h
        handle = lm.model.layers[L].register_forward_hook(hook)
        try:
            h = lm.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos).last_hidden_state[:, -1].float()
        finally:
            handle.remove()
        logp = torch.log_softmax(h @ W32.T, dim=-1)
        b = (logp[:, YES] - logp[:, NO]).tolist()
        res += [np.nan if it is None else v for it, v in zip(items, b)]
    return np.array(res), skipped


EVP = P[P.scenario_id.isin(VALID)].reset_index(drop=True)
other = {s: VALID[(VALID.index(s) + 1) % len(VALID)] for s in VALID}
lookup = {(r.scenario_id, r.variant): i for i, r in P.iterrows()}
U = P.loc[[lookup[(other[r.scenario_id], r.variant)] for _, r in EVP.iterrows()]]
q0 = EVP.q_c.tolist()
t0 = time.time()
SRC = {"true": record(EVP.see.tolist(), q0, EVP.span_see.tolist(), EVP.wspan_see.tolist()),
       "false": record(EVP.false.tolist(), q0, EVP.span_false.tolist(), EVP.wspan_false.tolist()),
       "unrelated": record(U.see.tolist(), U.q_c.tolist(), U.span_see.tolist(), U.wspan_see.tolist())}
print(f"recorded sources ({(time.time() - t0) / 60:.1f} min); {len(EVP)} items")

TARGETS = {"base": (EVP.base.tolist(), EVP.span_base.tolist(), EVP.wspan_base.tolist()),
           "A_only": (EVP.A_only.tolist(), EVP.span_A_only.tolist(), EVP.wspan_A_only.tolist())}
unpatched = {t: {p: belief(TARGETS[t][0], EVP[q].tolist(), TARGETS[t][1]) for p, q in PROBES.items()} for t in TARGETS}
for li, L in enumerate(LAYERS):
    path = f"{OUT}/{TAG}_sweep_layer{L:02d}.csv"
    if os.path.exists(path):
        print("skip layer", L); continue
    rows = []
    for target, kinds in [("base", KINDS), ("A_only", ["true"])]:
        st, sp, ws = TARGETS[target]
        for kind in kinds:
            r = EVP[["scenario_id", "item_id"]].copy()
            r["layer"], r["target"], r["kind"] = L, target, kind
            for p, q in PROBES.items():
                b, _ = belief_patch(st, EVP[q].tolist(), sp, ws, SRC[kind], li, MODE)
                r[f"raw_{p}"] = b
                r[f"d{p}"] = b - unpatched[target][p]
            rows.append(r)
    pd.concat(rows).to_csv(path, index=False)
    print(f"layer {L} done ({(time.time() - t0) / 60:.1f} min)")

## Step 3: results

In [ ]:
rng = np.random.default_rng(0)
def boot_median(x, n=2000):
    x = pd.Series(x).dropna().values
    if len(x) < 3:
        return np.nan, np.nan
    m = [np.median(rng.choice(x, len(x))) for _ in range(n)]
    return np.percentile(m, 2.5), np.percentile(m, 97.5)

sw = pd.concat(pd.read_csv(f) for f in sorted(glob.glob(f"{OUT}/{TAG}_sweep_layer*.csv")))
sw["dfill"] = sw[[f"df{i}" for i in range(len(FILLERS))]].mean(1)
sw["fill_raw"] = sw[[f"raw_f{i}" for i in range(len(FILLERS))]].mean(1)
for p in ["b", "c"]:
    sw[f"d{p}C"] = sw[f"d{p}"] - sw.dfill
base = sw[sw.target == "base"]
g = base.groupby(["scenario_id", "layer", "kind"])[["dbC", "dcC", "dfill"]].mean().reset_index()

# pass table (C question is the manipulation check)
rows = []
for L, grp in g.groupby("layer"):
    t = grp[grp.kind == "true"].set_index("scenario_id"); u = grp[grp.kind == "unrelated"].set_index("scenario_id")
    f = grp[grp.kind == "false"].set_index("scenario_id")
    diff = (t.dcC - u.dcC).dropna()
    rows.append(dict(layer=L, dcC_true=t.dcC.mean(), dcC_unrel=u.dcC.mean(), dcC_false=f.dcC.mean(), true_minus_unrel=diff.mean(),
                     share_true_wins=(diff > 0).mean(), dfill_true=t.dfill.mean(), dbC_true=t.dbC.mean(),
                     passes=bool((t.dcC.mean() >= 1) and (diff.mean() > 0.5) and ((diff > 0).mean() >= 0.7)
                                 and (abs(t.dfill.mean()) < 0.5) and (f.dcC.mean() < 0))))
res = pd.DataFrame(rows); res.to_csv(f"{OUT}/{TAG}_settings.csv", index=False)
display(res.round(2))

# D for colliders: r = dB / dC
t = g[g.kind == "true"].merge(sc[["dcC_see", "r_see", "r_do"]], left_on="scenario_id", right_index=True)
t["works"] = t.dcC >= 0.5 * t.dcC_see
t["r_edit"] = t.dbC / t.dcC
t["D"] = np.where(t.works, (t.r_see - t.r_edit) / (t.r_see - t.r_do), np.nan)
t.to_csv(f"{OUT}/{TAG}_scenario_scores.csv", index=False)

# explaining-away interaction under the patch (raw beliefs, filler-corrected)
pb = sw[sw.kind == "true"].assign(bC=lambda x: x.raw_b - x.fill_raw)
wide = pb.pivot_table(index=["scenario_id", "item_id", "layer"], columns="target", values="bC").reset_index()
r0 = ref.set_index("item_id")
wide["nb"] = wide.item_id.map(r0.b_base - fillm("base").set_axis(ref.item_id))
wide["na"] = wide.item_id.map(r0.b_A_only - fillm("A_only").set_axis(ref.item_id))
wide["EA_patch"] = (wide["base"] - wide["A_only"]) - (wide.nb - wide.na)
ea = wide.groupby(["scenario_id", "layer"]).EA_patch.mean().reset_index().merge(sc[["EA_text"]], left_on="scenario_id", right_index=True)
ea = ea.merge(t[["scenario_id", "layer", "works"]], on=["scenario_id", "layer"])
ea.to_csv(f"{OUT}/{TAG}_explaining_away.csv", index=False)

out = []
for L in LAYERS:
    tl, el = t[t.layer == L], ea[(ea.layer == L) & ea.works]
    dlo, dhi = boot_median(tl.D); elo, ehi = boot_median(el.EA_patch)
    out.append(dict(layer=L, works_rate=tl.works.mean(), n_scored=int(tl.D.notna().sum()), D_median=tl.D.median(), D_lo=dlo, D_hi=dhi,
                    EA_patch_median=el.EA_patch.median(), EA_lo=elo, EA_hi=ehi, EA_text_median=el.EA_text.median()))
summ = pd.DataFrame(out).merge(res[["layer", "passes"]], on="layer")
summ.to_csv(f"{OUT}/{TAG}_summary.csv", index=False)
display(summ.round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ok = summ[summ.n_scored >= 5]
axes[0].errorbar(ok.layer, ok.D_median, yerr=[ok.D_median - ok.D_lo, ok.D_hi - ok.D_median], fmt="o-", capsize=3)
axes[0].axhline(0, ls="--", c="k", lw=.5); axes[0].axhline(1, ls="--", c="k", lw=.5)
axes[0].set_title("Colliders: D of C patches (0 = seeing, 1 = doing)"); axes[0].set_xlabel("layer"); axes[0].set_ylabel("median D (95% CI)")
axes[1].errorbar(ok.layer, ok.EA_patch_median, yerr=[ok.EA_patch_median - ok.EA_lo, ok.EA_hi - ok.EA_patch_median], fmt="o-", capsize=3, label="C patched")
axes[1].axhline(sc.loc[VALID, "EA_text"].median(), color="C1", ls="--", label="C read in text")
axes[1].axhline(0, c="k", lw=.5); axes[1].set_title("Explaining-away interaction"); axes[1].set_xlabel("layer"); axes[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/{TAG}_results.png", dpi=150, bbox_inches="tight"); plt.show()